# Module 3 · PyTorch Fundamentals & The Deep Learning Gateway

**Track:** Main Track — Gateway 14  
**Toolchain:** PyTorch  
**Prerequisites:** ML Subtrack (13_01–13_08)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## Why This Notebook Exists

You have just completed the ML subtrack. You understand probability, optimization, regularization, and model evaluation in the classical sense. **Everything from this point forward runs on PyTorch.** The DL subtrack, the NLP subtrack, the model development notebook, alignment, inference optimization — all of it is PyTorch.

This notebook has two jobs:

1. **Part 1 (Code):** Teach you the PyTorch fundamentals you will use in every single notebook from here onward. This is not optional — it is a hands-on crash course.
2. **Part 2 (Routing):** Direct you through the DL and NLP subtracks before you resume the Main Track.

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat PyTorch as a black box — they copy-paste model code from tutorials and pray it trains. Seniors understand that PyTorch is a *computational graph engine*: every operation on a tensor is recorded into a directed acyclic graph (DAG), and `.backward()` walks that graph in reverse to compute gradients via the chain rule. If you understand the graph, you can debug *anything* — exploding gradients, memory leaks, incorrect loss curves.

**Mental Model:** Think of PyTorch as a tape recorder. When you do math with tensors that have `requires_grad=True`, PyTorch silently records every operation onto a tape. When you call `.backward()`, it rewinds the tape and computes the derivative of each operation in reverse. This is automatic differentiation — the engine behind all of deep learning.

**Common Junior Pitfall:** Forgetting `torch.no_grad()` during inference. Without it, PyTorch still records the computational graph (wasting memory and slowing down inference by 2-3×). Every production inference path MUST be wrapped in `with torch.no_grad():`.

---

## 📑 Table of Contents

* [Why This Notebook Exists](#why-this-notebook-exists)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Tensors, Shapes & Devices](#1-tensors-shapes-devices)
* [2. Autograd: The Computational Graph Engine](#2-autograd-the-computational-graph-engine)
* [3. nn.Module: Building Neural Networks](#3-nnmodule-building-neural-networks)
* [4. Loss Functions & Optimizers](#4-loss-functions-optimizers)
* [5. The Training Loop — From Scratch](#5-the-training-loop-from-scratch)
* [6. Dataset & DataLoader](#6-dataset-dataloader)
* [7. GPU Acceleration & Mixed Precision](#7-gpu-acceleration-mixed-precision)
* [8. End-to-End Project: Fashion-MNIST MLP](#8-end-to-end-project-fashion-mnist-mlp)
  * [8.1 Loading the Dataset with torchvision](#81-loading-the-dataset-with-torchvision)
  * [8.2 Defining the MLP](#82-defining-the-mlp)
  * [8.3 The Training Loop (Putting It All Together)](#83-the-training-loop-putting-it-all-together)
  * [8.4 Inspecting Predictions](#84-inspecting-predictions)
  * [8.5 What You Just Built](#85-what-you-just-built)
* [9. Saving & Loading Models](#8-saving-loading-models)
* [Mandatory: Deep Learning Subtrack](#mandatory-deep-learning-subtrack)
* [Highly Recommended: NLP Subtrack](#highly-recommended-nlp-subtrack)
* [Optional Specializations](#optional-specializations)
* [Your Path Forward](#your-path-forward)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: What happens if you forget `optimizer.zero_grad()`?](#q1-what-happens-if-you-forget-optimizerzero_grad)
  * [Q2: Why must you call `model.eval()` before inference?](#q2-why-must-you-call-modeleval-before-inference)
  * [Q3: You get `RuntimeError: Expected all tensors to be on the same device`. What went wrong?](#q3-you-get-runtimeerror-expected-all-tensors-to-be-on-the-same-device-what-went-wrong)
  * [Q4: Why save `model.state_dict()` instead of the full model?](#q4-why-save-modelstate_dict-instead-of-the-full-model)
  * [Q5: What is the difference between `torch.no_grad()` and `model.eval()`?](#q5-what-is-the-difference-between-torchno_grad-and-modeleval)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Shape Surgery](#exercise-1-shape-surgery)
  * [Exercise 2: Custom Dataset](#exercise-2-custom-dataset)
  * [Exercise 3: Full Training Pipeline](#exercise-3-full-training-pipeline)
  * [Exercise 4: Parameter Counting](#exercise-4-parameter-counting)


# Part 1 — PyTorch Fundamentals

## 1. Tensors, Shapes & Devices

A **tensor** is a multi-dimensional array — the atomic data structure of deep learning. Every input, every weight, every gradient is a tensor.

| Rank | Name | Shape Example | AI Use Case |
|:----:|------|:-------------:|-------------|
| 0 | Scalar | `()` | Loss value |
| 1 | Vector | `(768,)` | Single embedding |
| 2 | Matrix | `(32, 768)` | Batch of embeddings |
| 3 | 3D Tensor | `(32, 128, 768)` | Batch × Sequence × Hidden (Transformers) |
| 4 | 4D Tensor | `(32, 3, 224, 224)` | Batch × Channels × Height × Width (Vision) |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === 1.1 Creating Tensors ===
print("--- 1.1 Creating Tensors ---")

# From Python data
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(f"From list:    shape={x.shape}, dtype={x.dtype}")

# Common initializations (you'll use these constantly)
zeros = torch.zeros(2, 3)           # All zeros
ones = torch.ones(2, 3)             # All ones 
randn = torch.randn(2, 3)           # Normal distribution (mean=0, std=1)
arange = torch.arange(0, 10, 2)     # [0, 2, 4, 6, 8]

# === 1.2 Critical Shape Operations ===
print("\n--- 1.2 Shape Operations ---")
t = torch.randn(2, 3, 4)  # Shape: (batch, seq_len, features)

# view/reshape: Reinterpret dimensions (same data, new shape)
reshaped = t.view(2, 12)    # Flatten last two dims: (2, 3*4)
print(f"view(2,12):   {t.shape} → {reshaped.shape}")

# permute: Reorder dimensions (critical for attention)
permuted = t.permute(0, 2, 1)  # (batch, features, seq_len)
print(f"permute:      {t.shape} → {permuted.shape}")

# unsqueeze/squeeze: Add/remove dimensions of size 1
squeezed = torch.randn(1, 3, 1, 4).squeeze()  # Remove all size-1 dims
print(f"squeeze:      (1,3,1,4) → {squeezed.shape}")

# transpose: Swap two dimensions (shortcut for common permute)
transposed = t.transpose(1, 2)  # Swap seq_len and features
print(f"transpose:    {t.shape} → {transposed.shape}")

# @ operator: Matrix multiplication (the core of neural networks)
a = torch.randn(2, 3, 4)
b = torch.randn(2, 4, 5)
c = a @ b  # Batched matmul: (2, 3, 5)
print(f"matmul:       {a.shape} @ {b.shape} = {c.shape}")

# === 1.3 Devices (CPU vs GPU) ===
print("\n--- 1.3 Devices ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move tensor to device
x_gpu = x.to(device)
print(f"Tensor device: {x_gpu.device}")

# CRITICAL: You cannot mix devices!
# torch.randn(3, device='cuda') + torch.randn(3, device='cpu')  → RuntimeError
print("\n⚠️  Rule: ALL tensors in an operation must be on the SAME device.")
print("   model.to(device) and input.to(device) before forward pass.")

---

## 2. Autograd: The Computational Graph Engine

Autograd is the engine that makes deep learning possible. It automatically computes gradients (derivatives) of any computation.

**How it works:**
1. Create tensors with `requires_grad=True`
2. Do math — PyTorch records every operation into a DAG
3. Call `.backward()` on the output — PyTorch walks the DAG in reverse, computing gradients via the chain rule
4. Access gradients via `.grad`

This is *exactly* what happens when you train any model — from a 2-parameter linear regression to a 70B-parameter LLM.

In [ ]:
# === 2.1 Basic Autograd ===
print("--- 2.1 Basic Autograd ---")

# Simulate a single neuron: y = w*x + b, loss = (y - target)^2
w = torch.tensor([2.0], requires_grad=True)   # Weight
b = torch.tensor([1.0], requires_grad=True)   # Bias
x = torch.tensor([3.0])                       # Input (no grad needed)
target = torch.tensor([10.0])                  # Target

# Forward pass
y = w * x + b         # y = 2*3 + 1 = 7
loss = (y - target)**2  # loss = (7-10)^2 = 9

print(f"Forward: y={y.item():.1f}, loss={loss.item():.1f}")

# Backward pass — computes ALL gradients automatically
loss.backward()

# d(loss)/dw = 2*(y-target)*x = 2*(7-10)*3 = -18
# d(loss)/db = 2*(y-target)*1 = 2*(7-10)*1 = -6
print(f"Gradients: dL/dw={w.grad.item():.1f}, dL/db={b.grad.item():.1f}")
print(f"Verify:    dL/dw should be 2*(7-10)*3 = {2*(7-10)*3:.1f} ✅")

# === 2.2 Gradient Accumulation (important gotcha!) ===
print("\n--- 2.2 Gradient Accumulation ---")
# PyTorch ACCUMULATES gradients by default. You must zero them each step.
print(f"Before zero_grad: w.grad = {w.grad.item()}")
w.grad.zero_()
b.grad.zero_()
print(f"After zero_grad:  w.grad = {w.grad.item()}")
print("⚠️  If you forget optimizer.zero_grad(), gradients accumulate across batches!")

# === 2.3 torch.no_grad() — Disabling the tape recorder ===
print("\n--- 2.3 torch.no_grad() ---")
# During inference, you DON'T need gradients. Disabling them:
# - Saves ~30% memory (no graph stored)
# - Speeds up computation by ~20%
with torch.no_grad():
    y_inference = w * x + b
    print(f"Inference result: {y_inference.item():.1f}")
    print(f"Has grad_fn? {y_inference.grad_fn}")

# Compare: WITH grad tracking
y_training = w * x + b
print(f"Training result:  {y_training.item():.1f}")
print(f"Has grad_fn? {y_training.grad_fn}")

print("\n📌 Rule: ALWAYS wrap inference in torch.no_grad() or @torch.inference_mode()")

---

## 3. nn.Module: Building Neural Networks

`nn.Module` is the base class for *everything* in PyTorch — layers, models, loss functions. Understanding it is non-negotiable.

**The contract:**
1. Define layers in `__init__`
2. Define the computation in `forward`
3. PyTorch auto-discovers all parameters via `self.parameters()`
4. Never call `forward()` directly — call the module itself: `model(x)` (which triggers hooks and autograd)

In [ ]:
# === 3.1 Common Building Blocks ===
print("--- 3.1 Common Layers (used everywhere in the curriculum) ---")

# nn.Linear: y = xW^T + b (the core of every MLP, attention projection)
linear = nn.Linear(in_features=768, out_features=256)
print(f"nn.Linear(768→256): W={linear.weight.shape}, b={linear.bias.shape}")

# nn.Embedding: Lookup table (tokenizer IDs → dense vectors)
# Used in: Word2Vec (NB16_02), Transformers (NB15_06), LLMs
embedding = nn.Embedding(num_embeddings=50000, embedding_dim=768)
token_ids = torch.tensor([42, 1337, 7])  # 3 token IDs
vectors = embedding(token_ids)  # Shape: (3, 768)
print(f"nn.Embedding(50K vocab, 768d): input={token_ids.shape} → output={vectors.shape}")

# nn.LayerNorm: Normalize across features (used in every Transformer layer)
layer_norm = nn.LayerNorm(768)
print(f"nn.LayerNorm(768): normalizes the last dimension")

# nn.Dropout: Randomly zero elements during training (regularization)
dropout = nn.Dropout(p=0.1)
print(f"nn.Dropout(0.1): drops 10% of activations during training")

# === 3.2 Building a Custom Module ===
print("\n--- 3.2 Custom nn.Module ---")

class TransformerMLP(nn.Module):
    """The MLP block inside every Transformer layer.
    
    This exact pattern appears in:
    - NB15_06 (Transformer architecture)
    - NB16_05 (BERT)
    - NB16_07 / 15_model_development (LoRA targets these layers)
    """
    def __init__(self, d_model=768, d_ff=3072, dropout=0.1):
        super().__init__()  # MUST call parent __init__
        self.linear1 = nn.Linear(d_model, d_ff)     # Up-project
        self.linear2 = nn.Linear(d_ff, d_model)     # Down-project
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Pre-norm residual connection (modern Transformer pattern)
        residual = x
        x = self.norm(x)
        x = self.linear1(x)
        x = F.gelu(x)          # GELU activation (used in GPT-2, BERT, Llama)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x + residual     # Residual connection

mlp = TransformerMLP()
test_input = torch.randn(2, 10, 768)  # (batch, seq_len, d_model)
output = mlp(test_input)
print(f"TransformerMLP: input={test_input.shape} → output={output.shape}")

# === 3.3 Parameter Inspection ===
print("\n--- 3.3 Parameter Inspection ---")
total_params = sum(p.numel() for p in mlp.parameters())
trainable_params = sum(p.numel() for p in mlp.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nNamed parameters:")
for name, param in mlp.named_parameters():
    print(f"  {name:20s} shape={str(list(param.shape)):15s} trainable={param.requires_grad}")

# === 3.4 Freezing Parameters (critical for LoRA, transfer learning) ===
print("\n--- 3.4 Freezing Parameters ---")
# In fine-tuning (NB16_07), you freeze the base model and train only adapters
for param in mlp.parameters():
    param.requires_grad = False  # Freeze everything
# Then add trainable adapters on top
mlp.adapter = nn.Linear(768, 768)  # Only this is trained

frozen = sum(p.numel() for p in mlp.parameters() if not p.requires_grad)
unfrozen = sum(p.numel() for p in mlp.parameters() if p.requires_grad)
print(f"Frozen: {frozen:,}  |  Trainable: {unfrozen:,} ({unfrozen/(frozen+unfrozen):.1%})")
print("This is exactly what LoRA does — freeze 99.9%, train 0.1%")

---

## 4. Loss Functions & Optimizers

Every training loop needs two things: a **loss function** (tells the model how wrong it is) and an **optimizer** (tells the model how to fix itself).

| Loss Function | Use Case | Curriculum Usage |
|--------------|----------|------------------|
| `nn.MSELoss` | Regression | NB13 (linear models), NB15_01 |
| `nn.CrossEntropyLoss` | Classification (multi-class) | NB15_07 (GNNs), NB16_05 (BERT) |
| `nn.BCEWithLogitsLoss` | Binary classification | NB15_01 (backprop demo) |
| `nn.NLLLoss` | When you apply log_softmax yourself | NB16_02 (Word2Vec) |

| Optimizer | Use Case | Curriculum Usage |
|----------|----------|------------------|
| `SGD` | Baseline, good for convex problems | NB15_04 |
| `Adam` | Default for most DL | NB15_04, NB16_03 |
| `AdamW` | Adam + decoupled weight decay | NB15_04, NB16_07 (fine-tuning), production default |

In [ ]:
# === 4.1 Loss Functions ===
print("--- 4.1 Loss Functions ---")

# Classification: CrossEntropyLoss
# NOTE: CrossEntropyLoss expects RAW logits, NOT softmax probabilities!
logits = torch.tensor([[2.0, 1.0, 0.1]])   # Model output (3 classes)
target = torch.tensor([0])                   # True class = 0
ce_loss = nn.CrossEntropyLoss()
loss_val = ce_loss(logits, target)
print(f"CrossEntropyLoss: logits={logits.tolist()}, target={target.item()} → loss={loss_val.item():.4f}")

# Regression: MSELoss
prediction = torch.tensor([3.5])
target_reg = torch.tensor([4.0])
mse_loss = nn.MSELoss()
loss_val = mse_loss(prediction, target_reg)
print(f"MSELoss: pred={prediction.item()}, target={target_reg.item()} → loss={loss_val.item():.4f}")

# === 4.2 Optimizers ===
print("\n--- 4.2 Optimizers ---")

model = nn.Linear(10, 2)

# AdamW is the production default for Transformers
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,              # Learning rate
    weight_decay=0.01,     # L2 regularization (decoupled in AdamW)
)

print(f"Optimizer: AdamW, lr=1e-3, weight_decay=0.01")
print(f"Parameters managed: {len(list(model.parameters()))}")

# Learning rate scheduler (cosine annealing — used in all modern training)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
print(f"Scheduler: CosineAnnealingLR (T_max=100 steps)")
print("\n📌 The optimizer.step() → scheduler.step() pattern is in every training loop.")

---

## 5. The Training Loop — From Scratch

This is the single most important pattern in deep learning. Every model — from a 2-layer MLP to GPT-4 — follows the exact same loop:

```
for each batch:
    1. Forward pass:    predictions = model(inputs)
    2. Compute loss:    loss = loss_fn(predictions, targets)
    3. Backward pass:   loss.backward()
    4. Update weights:  optimizer.step()
    5. Reset gradients: optimizer.zero_grad()
```

In [ ]:
# === Complete Training Loop ===
# Task: Learn y = 3x + 2 from data
import torch
import torch.nn as nn

# Generate synthetic data
torch.manual_seed(42)
X = torch.randn(200, 1)          # 200 data points
y = 3 * X + 2 + torch.randn(200, 1) * 0.3  # y = 3x + 2 + noise

# Model: single linear layer (should learn w=3, b=2)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

print(f"Before training: w={model.weight.item():.3f}, b={model.bias.item():.3f}")

# ===== THE TRAINING LOOP =====
for epoch in range(100):
    # 1. Forward pass
    predictions = model(X)
    
    # 2. Compute loss
    loss = loss_fn(predictions, y)
    
    # 3. Backward pass (compute gradients)
    loss.backward()
    
    # 4. Update weights
    optimizer.step()
    
    # 5. RESET GRADIENTS (critical — forgetting this is the #1 bug)
    optimizer.zero_grad()
    
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d}: loss={loss.item():.4f}, w={model.weight.item():.3f}, b={model.bias.item():.3f}")

print(f"\nAfter training:  w={model.weight.item():.3f}, b={model.bias.item():.3f}")
print(f"Expected:        w=3.000, b=2.000")
print(f"\n✅ This 5-step loop is the foundation of ALL training in this curriculum.")

---

## 6. Dataset & DataLoader

In production, you never feed the entire dataset at once. You use:
- **`Dataset`**: Defines how to load a single sample
- **`DataLoader`**: Wraps Dataset and provides batching, shuffling, and parallel loading

This pattern appears in every training notebook from NB15 onward.

In [ ]:
from torch.utils.data import Dataset, DataLoader

# === 6.1 Custom Dataset ===
class TextClassificationDataset(Dataset):
    """Example: token IDs → sentiment label.
    This pattern is used in NB16_02 (Word2Vec), NB16_05 (BERT), etc.
    """
    def __init__(self, num_samples=1000, seq_len=32, vocab_size=5000):
        self.token_ids = torch.randint(0, vocab_size, (num_samples, seq_len))
        self.labels = torch.randint(0, 2, (num_samples,))  # Binary sentiment
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # Must return a dict or tuple per sample
        return self.token_ids[idx], self.labels[idx]

# === 6.2 DataLoader ===
dataset = TextClassificationDataset(num_samples=1000)
dataloader = DataLoader(
    dataset,
    batch_size=32,          # Number of samples per batch
    shuffle=True,           # Randomize order each epoch
    num_workers=0,          # Parallel data loading (set >0 in production)
    drop_last=True,         # Drop last incomplete batch
)

print(f"Dataset size:  {len(dataset)}")
print(f"Batch size:    32")
print(f"Batches/epoch: {len(dataloader)}")

# Iterate one batch
batch_tokens, batch_labels = next(iter(dataloader))
print(f"\nBatch tokens shape: {batch_tokens.shape}")
print(f"Batch labels shape: {batch_labels.shape}")
print(f"Sample tokens: {batch_tokens[0][:10].tolist()}...")
print(f"Sample label:  {batch_labels[0].item()}")

print("\n📌 The DataLoader handles batching, shuffling, and multiprocessing.")
print("   You just define __len__ and __getitem__.")

---

## 7. GPU Acceleration & Mixed Precision

Moving computation to GPU is the single biggest speedup in deep learning (10-100× faster). Mixed precision training (bf16/fp16) halves memory usage and nearly doubles throughput.

| Precision | Bits | VRAM | Speed | Used For |
|-----------|:----:|:----:|:-----:|----------|
| `float32` | 32 | 1× | 1× | Default (CPU) |
| `float16` | 16 | 0.5× | ~2× | Training with AMP |
| `bfloat16` | 16 | 0.5× | ~2× | **Production default** (better range than fp16) |
| `int8` | 8 | 0.25× | ~3× | Inference quantization (NB30) |
| `int4` / `nf4` | 4 | 0.125× | ~4× | QLoRA (NB16_07) |

In [ ]:
# === 7.1 The .to(device) Pattern ===
print("--- 7.1 Device Management ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The production pattern: move BOTH model and data to device
model = nn.Linear(768, 256).to(device)
x = torch.randn(8, 768).to(device)  # Batch of 8
output = model(x)
print(f"Model device: {next(model.parameters()).device}")
print(f"Input device: {x.device}")
print(f"Output shape: {output.shape}")

# === 7.2 Automatic Mixed Precision (AMP) ===
print("\n--- 7.2 Mixed Precision Training ---")
# This is the production pattern used in NB16_06, NB16_07, and all fine-tuning

amp_code = """
# Production mixed precision training pattern:
scaler = torch.amp.GradScaler()  # Prevents underflow in fp16 gradients

for batch in dataloader:
    optimizer.zero_grad()
    
    # autocast: run forward pass in fp16/bf16 (2x faster, half memory)
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        output = model(batch)
        loss = loss_fn(output, targets)
    
    scaler.scale(loss).backward()   # Scale loss to prevent gradient underflow
    scaler.step(optimizer)          # Unscale and step
    scaler.update()                 # Update scale factor
"""
print(amp_code)
print("📌 bf16 is preferred over fp16 on modern GPUs (A100, H100, RTX 4090).")
print("   It has the same range as fp32 but half the precision — no need for GradScaler.")

---

## 8. End-to-End Project: Fashion-MNIST MLP

Everything above taught you individual tools — tensors, autograd, modules, losses, optimizers, dataloaders, and GPU patterns. Now we assemble **all of them** into a complete, production-style training pipeline on a real dataset.

**Why Fashion-MNIST?** It is a drop-in MNIST replacement with 10 clothing categories (T-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot). Each image is 28×28 grayscale. It is small enough to train on CPU in under a minute, yet complex enough that a random baseline only gets 10% accuracy.

> **Bridge to NB15_01:** After you see the model learn below, the DL subtrack will show you *exactly* what happens inside `.backward()` — the chain rule, Jacobians, and gradient flow. You will no longer treat autograd as magic.


### 8.1 Loading the Dataset with `torchvision`

`torchvision.datasets` provides common benchmarks. The transform pipeline converts PIL images to tensors and normalizes pixel values to mean=0.5, std=0.5 (mapping [0,1] → [-1,1]).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# === Data pipeline ===
transform = transforms.Compose([
    transforms.ToTensor(),                          # PIL → Tensor [0, 1]
    transforms.Normalize(mean=(0.5,), std=(0.5,)),  # [0, 1] → [-1, 1]
])

train_dataset = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Quick sanity check
images, labels = next(iter(train_loader))
print(f"Batch shape : {images.shape}")    # (64, 1, 28, 28)
print(f"Label shape : {labels.shape}")     # (64,)
print(f"Pixel range : [{images.min():.1f}, {images.max():.1f}]")
print(f"Classes     : {len(CLASS_NAMES)}")
print(f"Train size  : {len(train_dataset):,}")
print(f"Test size   : {len(test_dataset):,}")


### 8.2 Defining the MLP

Our MLP has three fully-connected layers. `nn.Flatten()` converts each `(1, 28, 28)` image into a `(784,)` vector. We use ReLU activations and Dropout for regularization — the same building blocks you learned in Sections 3–5.

| Layer | Input → Output | Parameters | Purpose |
|-------|:--------------:|:----------:|:--------|
| Flatten | (1,28,28) → (784,) | 0 | Vectorize the image |
| Linear 1 | 784 → 256 | 200,960 | Feature extraction |
| Linear 2 | 256 → 128 | 32,896 | Compression |
| Linear 3 (head) | 128 → 10 | 1,290 | Classification logits |
| **Total** | | **235,146** | |


In [ ]:
class FashionMLP(nn.Module):
    """A 3-layer MLP for Fashion-MNIST classification.
    
    This is your first real model. Every pattern here — __init__ for layers,
    forward() for computation, residual-free sequential flow — will be
    extended in the DL subtrack (15_01 onward).
    """
    def __init__(self, input_dim=784, hidden1=256, hidden2=128, num_classes=10, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten()                     # (1, 28, 28) → (784,)
        self.layer1  = nn.Linear(input_dim, hidden1)    # 784 → 256
        self.layer2  = nn.Linear(hidden1, hidden2)      # 256 → 128
        self.head    = nn.Linear(hidden2, num_classes)  # 128 → 10
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.flatten(x)           # Vectorize
        x = F.relu(self.layer1(x))    # Hidden 1 + activation
        x = self.dropout(x)           # Regularization
        x = F.relu(self.layer2(x))    # Hidden 2 + activation
        x = self.dropout(x)
        x = self.head(x)              # Raw logits (no softmax!)
        return x

# Instantiate & inspect
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FashionMLP().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model on    : {device}")
print(f"Parameters  : {total_params:,}")
print()
for name, p in model.named_parameters():
    print(f"  {name:18s}  shape={str(list(p.shape)):14s}  params={p.numel():>7,}")


### 8.3 The Training Loop (Putting It All Together)

This is the exact 5-step loop from Section 5, but now operating on real batches from a DataLoader, on a real model, with a real loss function. Watch the loss decrease and accuracy climb — this is deep learning working.


In [ ]:
# === Hyperparameters ===
EPOCHS = 5
LR = 1e-3

loss_fn   = nn.CrossEntropyLoss()    # Expects raw logits, NOT softmax
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# === Training Function ===
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()  # Enable dropout
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)  # Step 0: Move to device
        
        # Step 1: Forward pass
        logits = model(images)
        
        # Step 2: Compute loss
        loss = loss_fn(logits, labels)
        
        # Step 3: Backward pass
        loss.backward()
        
        # Step 4: Update weights
        optimizer.step()
        
        # Step 5: Zero gradients (for next iteration)
        optimizer.zero_grad()
        
        # Track metrics
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    
    return running_loss / total, correct / total

# === Validation Function ===
# NOTE: model.eval() + torch.no_grad() — BOTH are required for correct inference.
@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()  # Disable dropout, switch BatchNorm to eval
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = loss_fn(logits, labels)
        
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    
    return running_loss / total, correct / total

# === Main Training Loop ===
print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Test Loss':>9}  {'Test Acc':>8}")
print("-" * 52)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_loss, test_acc   = evaluate(model, test_loader, loss_fn, device)
    
    print(f"{epoch:5d}  {train_loss:10.4f}  {train_acc:8.1%}  {test_loss:9.4f}  {test_acc:7.1%}")

print(f"\n✅ Final test accuracy: {test_acc:.1%}")
print(f"   (Random baseline = 10.0% — your MLP should reach ~87-89%)")


### 8.4 Inspecting Predictions

Always look at what the model actually predicts. Below we grab a test batch, run inference, and display results. This is a habit that separates engineers from script-runners.


In [ ]:
import random

# Grab a test batch
model.eval()
test_images, test_labels = next(iter(test_loader))
test_images, test_labels = test_images.to(device), test_labels.to(device)

with torch.no_grad():
    logits = model(test_images)
    probs  = F.softmax(logits, dim=1)           # Convert logits → probabilities
    preds  = logits.argmax(dim=1)                # Predicted class

# Show 8 random predictions
print(f"{'Index':>5}  {'True Label':>18}  {'Predicted':>18}  {'Confidence':>10}  {'Correct':>7}")
print("-" * 68)

indices = random.sample(range(len(test_labels)), 8)
for idx in indices:
    true_label = CLASS_NAMES[test_labels[idx].item()]
    pred_label = CLASS_NAMES[preds[idx].item()]
    confidence = probs[idx, preds[idx]].item()
    is_correct = '✅' if preds[idx] == test_labels[idx] else '❌'
    print(f"{idx:5d}  {true_label:>18}  {pred_label:>18}  {confidence:9.1%}  {is_correct:>7}")

# Per-class accuracy breakdown
print(f"\n{'Class':>18}  {'Accuracy':>8}")
print("-" * 30)
for cls_idx, name in enumerate(CLASS_NAMES):
    mask = test_labels == cls_idx
    if mask.sum() > 0:
        cls_acc = (preds[mask] == test_labels[mask]).float().mean().item()
        print(f"{name:>18}  {cls_acc:7.1%}")


### 8.5 What You Just Built

Congratulations — you have just built and trained a complete neural network from scratch. Let's map every component back to the PyTorch fundamentals you learned above:

| What You Did | Section It Came From |
|:-------------|:--------------------|
| `torch.tensor`, shapes, `.to(device)` | §1 Tensors, §7 GPU |
| `loss.backward()` computed all gradients | §2 Autograd |
| `class FashionMLP(nn.Module)` | §3 nn.Module |
| `nn.CrossEntropyLoss()`, `torch.optim.Adam` | §4 Losses & Optimizers |
| The 5-step train loop | §5 Training Loop |
| `DataLoader(dataset, batch_size=64, shuffle=True)` | §6 Dataset & DataLoader |
| `model.eval()`, `torch.no_grad()` | §2 + §3 |

**Next step → NB15_01** will open the black box of `.backward()` and show you the *exact* calculus that computed those gradients. You will derive backpropagation by hand using the chain rule and Jacobian matrices. The model you just trained is the intuition anchor for that math.

---


---

## 9. Saving & Loading Models

Two patterns exist. Use `state_dict` — never pickle the entire model object.

| Method | Saves | Portable? | Use Case |
|--------|-------|:---------:|----------|
| `torch.save(model.state_dict(), ...)` | Weights only | ✅ Yes | **Production default** |
| `torch.save(model, ...)` | Entire object | ❌ Fragile | Never use in production |

In [ ]:
import tempfile, os

print("--- 8. Saving & Loading ---")

# Create and train a simple model
model = TransformerMLP(d_model=768, d_ff=3072)

# SAVE: state_dict only (weights + bias values)
save_path = os.path.join(tempfile.gettempdir(), "model_weights.pt")
torch.save(model.state_dict(), save_path)
file_size_mb = os.path.getsize(save_path) / 1e6
print(f"Saved state_dict to {save_path} ({file_size_mb:.1f} MB)")

# LOAD: Create model architecture first, then load weights into it
loaded_model = TransformerMLP(d_model=768, d_ff=3072)  # Same architecture!
loaded_model.load_state_dict(torch.load(save_path, weights_only=True))
loaded_model.eval()  # Set to inference mode (disables dropout)

# Verify weights match
test_x = torch.randn(1, 5, 768)
with torch.no_grad():
    original_out = model.eval()(test_x)
    loaded_out = loaded_model(test_x)
    diff = (original_out - loaded_out).abs().max().item()
    print(f"Max diff between original and loaded: {diff:.10f}")
    print(f"✅ Weights loaded correctly!")

# Checkpoint pattern (saves training state for resume)
print("\n📌 For training checkpoints (resume after crash):")
checkpoint_code = """
# Save checkpoint
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}, 'checkpoint.pt')

# Load checkpoint
ckpt = torch.load('checkpoint.pt', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch'] + 1
"""
print(checkpoint_code)

# Cleanup
os.remove(save_path)

---

# Part 2 — The Subtrack Routing Map

## Mandatory: Deep Learning Subtrack

**You cannot proceed to the Main Track resume (NB16_07) without completing the DL subtrack.** These notebooks build the mathematical foundations that everything else sits on.

Complete these **8 notebooks** in sequence:

| # | Notebook | What You Will Learn | PyTorch Patterns Used |
|---|----------|--------------------|-----------------------|
| 01 | `05_01_neural_network_mathematics.ipynb` | Forward/backward pass, backprop from scratch | NumPy (no PyTorch yet — understand the raw math) |
| 02 | `05_02_activation_functions_and_gradients.ipynb` | ReLU, GELU, vanishing/exploding gradients | `F.relu`, `F.gelu`, autograd |
| 03 | `05_03_weight_initialization_theory.ipynb` | Xavier, Kaiming, why init matters | `nn.init.*`, `torch.no_grad` |
| 04 | `05_04_advanced_optimizers_and_loss_landscapes.ipynb` | SGD → Adam → AdamW, LR schedules | `torch.optim.*`, schedulers |
| 05 | `05_05_regularization_and_normalization.ipynb` | Dropout, BatchNorm, LayerNorm, weight decay | `nn.Dropout`, `nn.LayerNorm` |
| 06 | `20_transformer_architecture.ipynb` | Self-attention, GQA, RoPE, KV-Cache | Full `nn.Module`, `einsum`, masking |
| 07 | `06_graph_neural_networks.ipynb` | GCN, GraphSAGE, GAT, message passing | `nn.Module`, sparse tensors |
| 08 | `22_mixture_of_experts.ipynb` | MoE layers, top-K routing, load balancing | `nn.Module`, gating networks |

**Estimated time:** 15–25 hours.

---

## Highly Recommended: NLP Subtrack

The NLP subtrack bridges the gap between raw Transformers and real LLMs. Without it, you will use HuggingFace APIs without understanding what they do.

| # | Notebook | What You Will Learn |
|---|----------|--------------------|
| 01 | `19_01_text_preprocessing_tokenization.ipynb` | BPE/WordPiece tokenization, vocab |
| 02 | `19_02_word_embeddings_and_word2vec.ipynb` | Word2Vec, GloVe, embedding spaces |
| 03 | `19_03_recurrent_networks_and_lstms.ipynb` | RNNs, LSTMs, GRUs |
| 04 | `19_04_sequence_to_sequence_and_attention.ipynb` | Encoder-decoder, attention, beam search |
| 05 | `21_pretraining_objectives_and_bert.ipynb` | MLM, BERT, GPT pretraining |
| 06 | `23_01_huggingface_pipelines_and_transfer_learning.ipynb` | HuggingFace Trainer, transfer learning, AMP |

**Estimated time:** 15–25 hours.

---

## Optional Specializations

These are self-contained domain tracks. Take them based on your career goals:

| Subtrack | Location | Notebooks | Best For |
|----------|----------|:---------:|----------|
| 🔵 **Computer Vision** | `specialization_computer_vision/` | 17 | Vision, multimodal AI, document AI |
| 🟣 **Reinforcement Learning** | `specialization_reinforcement_learning/` | 6 | Alignment (RLHF/PPO), robotics, gaming |
| ⚪ **Recommender Systems** | `specialization_recommender_systems/` | 4 | E-commerce, media, social platforms |
| ⚪ **Time Series** | `specialization_time_series/` | 4 | FinTech, IoT, energy, demand forecasting |

---

## Your Path Forward

```
   YOU ARE HERE (Gateway 14)
        │
        ▼
   ┌──────────────────────────────────────────┐
   │  ⛔ MANDATORY: DL Subtrack (8 notebooks) │
   │  15_01 → 15_08                           │
   │  Neural networks → Transformers → MoE    │
   └──────────────────┬───────────────────────┘
                      │
                      ▼
   ┌──────────────────────────────────────────┐
   │  🟢 NLP Subtrack (6 notebooks)           │
   │  16_01 → 16_06                           │
   │  Tokenization → BERT → HuggingFace      │
   └──────────────────┬───────────────────────┘
                      │
                      ▼
   ┌──────────────────────────────────────────┐
   │  ✅ MAIN TRACK RESUMES                    │
   │  → 23_02_model_development.ipynb         │
   │  LoRA, QLoRA, SFT, experiment tracking   │
   └──────────────────────────────────────────┘
```

---

## ✅ Knowledge Check

### Q1: What happens if you forget `optimizer.zero_grad()`?
<details><summary>👉 Answer</summary>

Gradients **accumulate** across batches. If batch 1 computes dL/dw = 5 and batch 2 computes dL/dw = 3, without zeroing, the gradient becomes 8 instead of 3. The model takes larger and larger steps, causing unstable training. This is the #1 PyTorch beginner bug.
</details>

### Q2: Why must you call `model.eval()` before inference?
<details><summary>👉 Answer</summary>

`model.eval()` changes the behavior of two layer types: **Dropout** stops dropping neurons (uses full network), and **BatchNorm** uses running statistics instead of batch statistics. Without `.eval()`, inference results will be noisy and non-deterministic. Always pair with `torch.no_grad()` for optimal inference.
</details>

### Q3: You get `RuntimeError: Expected all tensors to be on the same device`. What went wrong?
<details><summary>👉 Answer</summary>

You have tensors on different devices (e.g., model on GPU, input on CPU). Fix: use `input = input.to(device)` and `model = model.to(device)` with the SAME device. This is the #2 most common PyTorch error. Define `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')` once and use it everywhere.
</details>

### Q4: Why save `model.state_dict()` instead of the full model?
<details><summary>👉 Answer</summary>

Saving the full model with `torch.save(model, ...)` uses Python's `pickle`, which creates a tight coupling between the saved file and your exact code structure. If you rename a class, move a file, or change an import, loading breaks. `state_dict()` saves only the tensor values (a plain dictionary), which is portable and can be loaded into any compatible architecture.
</details>

### Q5: What is the difference between `torch.no_grad()` and `model.eval()`?
<details><summary>👉 Answer</summary>

They do **different things** and you need **both** for inference:
- `model.eval()` — Changes layer behavior (disables dropout, switches BatchNorm to eval mode)
- `torch.no_grad()` — Disables gradient computation (saves memory, speeds up math)

Using only `model.eval()` still wastes memory building the computational graph. Using only `torch.no_grad()` means dropout is still active and BatchNorm uses batch stats.
</details>

---
## 🔨 Practical Practice

### Exercise 1: Shape Surgery
Given a tensor of shape `(8, 12, 64, 64)` (batch, heads, seq_len, head_dim), reshape it to `(8, 64, 768)` (batch, seq_len, d_model where d_model = heads × head_dim). This is the exact operation that happens after multi-head attention.

### Exercise 2: Custom Dataset
Create a `Dataset` class that loads a list of (sentence, label) pairs, tokenizes them with a simple word-to-index mapping, pads sequences to `max_len=50`, and returns `(token_ids_tensor, label_tensor)`. Wrap it in a DataLoader with batch_size=16.

### Exercise 3: Full Training Pipeline
Build and train a 2-layer MLP classifier on the `make_moons` dataset from sklearn. Use:
- `nn.Module` with `nn.Linear`, `F.relu`, `nn.Dropout`
- `nn.CrossEntropyLoss` and `torch.optim.AdamW`
- A proper train/eval split and `model.eval()` + `torch.no_grad()` for validation
- Print training loss and validation accuracy every 10 epochs

### Exercise 4: Parameter Counting
Write a function `count_parameters(model)` that prints a table of every parameter group (name, shape, trainable, count). Then apply it to a model where some layers are frozen and some are not (simulating LoRA). Verify the trainable percentage.

---

**Begin now →** `05_01_neural_network_mathematics.ipynb`